<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Attention step by step

**[Attention step by step](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/)** · Deep Learning and Transformers Explained · Module 5, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute attention by hand with PyTorch (scores, scaling, softmax, weighted sum), check every shape, trace the module practice, and read the attention weights of a small transformer trained on Larkfield's tickets, together with a test of which words really change its answer.

| | |
|---|---|
| **Time** | About 35 minutes, including about 2 minutes of training |
| **Needs** | An internet connection for the first cells (about 2 MB of tickets). No account. Colab and Kaggle have PyTorch. |
| **You should know** | Embeddings, from [Tokens and embeddings](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/), and softmax, from [Loss](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M05-L03-attention-step-by-step/attention-step-by-step-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The worked example

Three words, "damaged on arrival", with made-up queries, keys and values of 2 numbers each, as in the lesson.

> **Predict.** Which word will "damaged" (row 1) read most?

Run the cell.

In [ ]:
import math
from torch import nn

Q = torch.tensor([[0.5, 1.5], [0.0, 0.0], [1.0, 0.0]])   # damaged, on, arrival
K = torch.tensor([[1.0, 0.0], [0.0, 0.5], [0.0, 2.0]])
V = torch.tensor([[2.0, 0.0], [0.0, 0.0], [0.0, 1.0]])
scores = Q @ K.T
weights = torch.softmax(scores / math.sqrt(2), dim=1)
out = weights @ V
show_shape("scores", scores)
print(weights.round(decimals=3))
print(out.round(decimals=3))
print("PyTorch agrees:", torch.allclose(out, nn.functional.scaled_dot_product_attention(Q, K, V)))

> **Check.** Row 1 of the weights is 0.124, 0.148, 0.728: "damaged" reads mostly "arrival", and its output is (0.248, 0.728). Every row adds up to 1.

## 2. Module practice: trace one attention calculation

> **Your turn.** Use the numbers below. Compute, for the reading word **word 1** only: `scores_1` (the three dot products), `weights_1` (after ÷ √2 and softmax) and `out_1` (the weighted sum of the values). Work with tensors, one step per line. Then compute the weights of **all** words in `all_weights`.

In [ ]:
Q2 = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K2 = torch.tensor([[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]])
V2 = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
scores_1 = K2 @ Q2[0]
weights_1 = torch.softmax(scores_1 / math.sqrt(2), dim=0)
out_1 = weights_1 @ V2
all_weights = torch.softmax(Q2 @ K2.T / math.sqrt(2), dim=1)
print(scores_1, weights_1.round(decimals=3), out_1.round(decimals=3))
print(all_weights.round(decimals=3))

In [ ]:
expect_hash('weights_1', [round(float(v), 3) for v in weights_1] if isinstance(weights_1, torch.Tensor) else None, 'f2fe7a27c980ead2')

In [ ]:
expect_hash('out_1', [round(float(v), 3) for v in out_1] if isinstance(out_1, torch.Tensor) else None, 'b884df3bdad49693')

In [ ]:
expect_hash('the shape of all_weights', tuple(all_weights.shape) if isinstance(all_weights, torch.Tensor) else None, '28e7165fc1adc5b4')

Look at row 3 of `all_weights`: the query (1, 1) gives the same score, 2, with every key, so its weights are equal (0.333 each), and attention becomes a plain average.

## 3. Shapes in the real model

The tiny transformer uses PyTorch's `nn.MultiheadAttention` with 64 numbers per token and 4 heads.

> **Your turn.** Predict the shapes first. Then make a random batch `x` of 8 tickets × 50 tokens × 64 numbers, run `attn(x, x, x, average_attn_weights=False)`, and store the shapes of its two results in `out_shape` and `weights_shape` (as tuples).

In [ ]:
torch.manual_seed(0)
attn = nn.MultiheadAttention(64, 4, batch_first=True)
x = torch.randn(8, 50, 64)
out, w = attn(x, x, x, average_attn_weights=False)
out_shape, weights_shape = tuple(out.shape), tuple(w.shape)
print(out_shape, weights_shape)

In [ ]:
expect('out_shape', out_shape, (8, 50, 64))

In [ ]:
expect('weights_shape', weights_shape, (8, 4, 50, 50))

Words in, words out: (8, 50, 64) in and out. The weights are (batch, heads, tokens, tokens): 2,500 weights per head and ticket.

## 4. Attention weights on a real ticket

The next cell trains the course's `TinyTransformer` for **5 epochs** (about 2 minutes on a free Colab CPU). The lesson's model trained for 20 epochs, so your weights and accuracy will differ from the lesson's explorer; the way to read them is the same.

In [ ]:
vocab = Vocab([r["text"] for r in train_rows])
ids_train, ids_valid = token_ids(train_rows, vocab), token_ids(valid_rows, vocab)
set_seed(0)
tf = TinyTransformer(len(vocab))
history = train(tf, ids_train, labels(train_rows), ids_valid, labels(valid_rows), epochs=5, log=print)
tf.eval()

`nn.TransformerEncoderLayer` does not return its attention weights. So the cell below walks through the two blocks by hand: it normalizes the input as the block does (`norm1`), asks the block's attention for its weights, and then runs the whole block to get the input of the next block.

Run the cell. It shows what the word "not" reads in block 2 of ticket T-65052 ("Not a delivery issue … My card was charged twice …").

In [ ]:
row = next(r for r in valid_rows if r["ticket_id"] == "T-65052")
ids = torch.tensor([vocab.encode(row["text"])])
tokens = tokenize(row["text"])
with torch.no_grad():
    h = tf.embed(ids) + tf.position(torch.arange(ids.shape[1]))
    for block in tf.blocks.layers:
        x = block.norm1(h)
        _, w = block.self_attn(x, x, x, need_weights=True, average_attn_weights=False)
        h = block(h)
    probs = tf(ids).softmax(1)[0]
show_shape("weights", w)
q = tokens.index("not")
for head in range(4):
    top = w[0, head, q].topk(3)
    print(f"block 2, head {head + 1}:", [(tokens[i], round(v.item(), 2)) for v, i in zip(top.values, top.indices)])
print("answer:", TEAMS[int(probs.argmax())], round(probs.max().item(), 3))

## 5. Which words change the answer?

Attention weights show where each word took its information from. They do not show which words decided the team. To test influence, hide one word at a time (replace it with `<unk>`, ID 1) and see how much the team's score drops below the best other team's score.

> **Predict.** Will the word with the largest attention also be the word whose removal hurts most?

In [ ]:
team = int(probs.argmax())

def lead(logits):
    other = logits.clone()
    other[:, team] = -1e9
    return logits[:, team] - other.max(1).values

with torch.no_grad():
    base = lead(tf(ids))[0].item()
    hidden = ids.repeat(len(tokens), 1)
    hidden[torch.arange(len(tokens)), torch.arange(len(tokens))] = 1
    drop = base - lead(tf(hidden))
received = w[0].mean(0).mean(0)   # block 2: mean over heads and reading words
print("most attention received:", [tokens[i] for i in received.topk(3).indices])
print("largest drop when hidden:", [tokens[i] for i in drop.topk(3).indices])

> **Check.** In the lesson's model, the two lists are "remove, twice, charge" and "charged, charge, remove". For this ticket they overlap; over all 600 validation tickets, the most attended word is the most influential one in only 12.2% of the tickets. Use attention to look inside the computation, and a test like this one to say which inputs influence an answer.

## Next

You can now compute attention by hand, check its shapes, and read a model's attention weights without treating them as reasons. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/) has the explorer with the lesson's model and the module check. Module 6 builds a whole transformer block around attention.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them. The worked-example numbers are made up.